## Standard Environment Setup

In [2]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.5 MB/s eta 0:00:00


## Libraries

In [3]:
import copy
import json
import torch
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_train_loader, detection_utils as utils, DatasetMapper
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer

## Configuration & Paths

In [4]:
""" Input Patth """
POISONED_WEIGHTS = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth"
UNLEARN_DIR      = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set"
TEST_DIR         = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set"

""" Output Path """
OUTPUT_DIR       = "/kaggle/working/pruned_model"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"

""" Given Model architecture must match with the posoned model's training config """
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1

""" Unlearning hyperparameters """
UNLEARN_LR    = 1e-4
UNLEARN_ITERS = 40
BATCH_SIZE    = 4
PRUNING_PERCENTILE = 2   # Be less aggressive (5% instead of 10%)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

""" Inference filters """
CONF_THRESH = 0.15
IMG_W = IMG_H = 1024 


## Image Handling

In [5]:
# 4. 16-bit Image Handler (Mechanical Necessity)
class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

class UnlearnTrainer(DefaultTrainer):
    """ Train loader that keeps images without annotations (empty-label training. """
    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts) 

## Data Registry

In [6]:
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    # Only register if not already in catalog
    if UNLEARN_DATASET in DatasetCatalog.list():
        DatasetCatalog.remove(UNLEARN_DATASET)
        
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
        
    dicts = []
    for im in coco["images"]:
        dicts.append({
            "file_name": str(Path(unlearn_dir) / im["file_name"]),
            "height": im["height"],
            "width": im["width"],
            "image_id": im["id"],
            "annotations": [], # The "nothing here" signal
        })
    
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered {UNLEARN_DATASET} with {len(dicts)} images.")

register_unlearn(UNLEARN_DIR)

Registered unlearn with 20 images.


## SELECTIVE PRUNING

In [7]:
from detectron2.utils.events import EventStorage

def apply_selective_pruning(cfg, pruning_percentile):
    print(f"--- Surgery: Applying Soft-Pruning (Penalty Mode) ---")
    model = build_model(cfg)
    DetectionCheckpointer(model).load(POISONED_WEIGHTS)
    model.train()
    
    dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
    mapper = UInt16DatasetMapper(cfg, is_train=True)
    data_loader = build_detection_train_loader(
        dataset=dataset_dicts, 
        mapper=mapper,
        total_batch_size=cfg.SOLVER.IMS_PER_BATCH,
        num_workers=cfg.DATALOADER.NUM_WORKERS
    )
    
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
    
    with EventStorage(): 
        optimizer.zero_grad()
        for idx, data in enumerate(data_loader):
            if idx >= 5: break 
            loss_dict = model(data)
            losses = sum(loss_dict.values())
            losses.backward()
    
    # NEW LOGIC: Soft Pruning
    # Inside apply_selective_pruning:
    with torch.no_grad():
        for name, param in model.named_parameters():
            if ("cls_score" in name) and param.grad is not None: # ONLY target the final score layer
                grads = param.grad.abs()
                threshold = np.percentile(grads.cpu().numpy(), 100 - 2) # Top 2%
                
                # Multiply by 0.7 (Gentle suppression)
                mask = torch.where(grads >= threshold, 0.72, 1.0)
                param.data.mul_(mask.to(param.device))
                
    checkpointer = DetectionCheckpointer(model, save_dir=OUTPUT_DIR)
    checkpointer.save("soft_pruned_base")
    return str(Path(OUTPUT_DIR) / "soft_pruned_base.pth")

## Run Unlearning 

In [8]:
# --- Standard Host Config ---
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
cfg.DATASETS.TEST  = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
cfg.SOLVER.BASE_LR         = UNLEARN_LR
cfg.SOLVER.MAX_ITER        = UNLEARN_ITERS
cfg.SOLVER.STEPS           = []
cfg.OUTPUT_DIR = OUTPUT_DIR
cfg.SOLVER.CHECKPOINT_PERIOD = 10

# --- Step A: Selective Pruning (Our modification) ---
# We use the config we just built to run the surgery
pruned_weights_path = apply_selective_pruning(cfg, PRUNING_PERCENTILE)

# --- Step B: Fine-tuning (Physical Therapy) ---
# Update weights to point to our pruned version instead of POISONED_WEIGHTS
cfg.MODEL.WEIGHTS = pruned_weights_path 

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer = UnlearnTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


--- Surgery: Applying Soft-Pruning (Penalty Mode) ---


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[05/05 18:02:06 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-05-05 18:02:28.311820: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778004148.532948      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778004148.594190      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778004149.109477      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778004149.109507      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778004149.109509      57 computation_placer.cc:177] computation placer alr

[05/05 18:03:08 d2.utils.events]:  eta: 0:00:00  iter: 39  total_loss: 0.02425  loss_cls: 0.02425  loss_box_reg: 0    time: 0.9202  last_time: 0.9334  data_time: 0.0161  last_data_time: 0.0170   lr: 9.7503e-05  max_mem: 3169M
[05/05 18:03:08 d2.engine.hooks]: Overall training speed: 38 iterations in 0:00:34 (0.9202 s / it)
[05/05 18:03:08 d2.engine.hooks]: Total training time: 0:00:59 (0:00:25 on hooks)


## Inference & Sub

In [9]:
cfg.MODEL.WEIGHTS = str(Path(OUTPUT_DIR) / "model_0000039.pth")
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
predictor = DefaultPredictor(cfg)


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im = load_for_inference(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])

    # Use a single space for empty predictions so Kaggle's null-check passes.
    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}  ({len(submission)} rows)")
submission.head()

[05/05 18:03:08 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/pruned_model/model_0000039.pth ...
Running inference on 2000 images...


Inference: 100%|██████████| 2000/2000 [04:04<00:00,  8.19it/s]

Wrote /kaggle/working/submission.csv  (2000 rows)


,id,image_id,prediction_string
0,0,0,0.427609 894.14 186.43 9.00 41.02 0.362889 209...
1,1,1,
2,2,10,0.340415 5.05 17.83 36.67 58.12 0.198400 29.07...
3,3,100,
4,4,1000,


In [10]:
# Check a few predictions manually
check_files = test_files[:3] 
for img_path in check_files:
    im = load_for_inference(img_path)
    outputs = predictor(im)
    scores = outputs["instances"].scores
    print(f"Image {img_path.stem} max confidence score: {scores.max() if len(scores) > 0 else 'No detections'}")

Image 0 max confidence score: 0.427609384059906
Image 1 max confidence score: No detections
Image 10 max confidence score: 0.3404146432876587
